<a href="https://colab.research.google.com/github/prof-atritiack/1CC-MLAM-regressao-linear-CP2-2sem/blob/main/Exemplo_Aula_07_1CCPJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regressão Linear

Este notebook apresenta uma introdução prática à **correlação** e à **regressão linear**.

A proposta é aprender a:

- observar a relação entre duas variáveis numéricas;
- calcular e interpretar a correlação;
- escolher uma variável de entrada (**X**) e uma variável que queremos estimar (**y**);
- criar um modelo de regressão linear;
- visualizar os dados e a reta ajustada;
- interpretar o coeficiente angular e o intercepto.
- apresentar as métricas do modelo.

Essas etapas servem como referência para a **Challenge Sprint 3**, na qual será necessário selecionar duas variáveis numéricas adequadas e desenvolver uma regressão linear em Python.

## 1. Bibliotecas utilizadas

No Google Colab, as bibliotecas abaixo já estão disponíveis.  
Não é necessário instalar pacotes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Métricas
# R2 = Coeficiente de determinação
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## 2. Dataset

Neste exemplo será utilizada a base **Consumo de Cerveja em São Paulo**.

A base possui informações como temperatura, chuva, final de semana e consumo de cerveja.

Para acompanhar o notebook, o arquivo `consumo.csv` deve estar disponível na sessão do Colab.

In [ ]:
dados = pd.read_csv("consumo.csv", sep=";")
dados.head(10)

In [ ]:
dados.tail(10)

## 3. Conhecendo os dados

Antes de criar um modelo, é importante observar quais informações existem na base.

In [ ]:
print("Quantidade de linhas e colunas:", dados.shape)
dados.info()

In [ ]:
# Histograma da coluna consumo
plt.figure(figsize=(8, 5))
plt.hist(dados["consumo"], bins=20, alpha=0.7)

In [ ]:
dados.describe()

## 4. Selecionando apenas as variáveis numéricas

A regressão linear trabalha com valores numéricos.  
Vamos observar as colunas numéricas disponíveis na base.

In [ ]:
dados_numericos = dados.select_dtypes(include="number")
dados_numericos.head()

# Correlação

A **correlação** ajuda a observar a relação linear entre duas variáveis.

O coeficiente de correlação varia de **-1 a +1**:

- valores próximos de **+1** indicam uma relação linear positiva forte;
- valores próximos de **-1** indicam uma relação linear negativa forte;
- valores próximos de **0** indicam uma relação linear fraca ou inexistente.

Uma correlação positiva indica que as duas variáveis tendem a crescer juntas.  
Uma correlação negativa indica que, quando uma aumenta, a outra tende a diminuir.

**Correlação não significa causalidade.**

## 5. Demonstração interativa de correlação

Use o controle abaixo para comparar exemplos de correlação positiva, negativa e sem correlação.

In [ ]:
from ipywidgets import interact

def mostrar_correlacao(tipo="Positiva"):
    np.random.seed(10)
    x = np.arange(1, 21)

    if tipo == "Positiva":
        y = 2 * x + np.random.normal(0, 5, len(x))
    elif tipo == "Negativa":
        y = 45 - 2 * x + np.random.normal(0, 5, len(x))
    else:
        y = np.random.normal(20, 8, len(x))

    r = np.corrcoef(x, y)[0, 1]

    plt.figure(figsize=(7, 4))
    plt.scatter(x, y)
    plt.xlabel("Variável X")
    plt.ylabel("Variável Y")
    plt.title(f"{tipo} | Correlação = {r:.2f}")
    plt.grid(alpha=0.2)
    plt.show()

interact(
    mostrar_correlacao,
    tipo=["Positiva", "Negativa", "Sem correlação"]
);

## 6. Correlação no dataset

Agora podemos calcular a correlação entre as variáveis numéricas da base.

In [ ]:
correlacoes = dados_numericos.corr()
correlacoes.round(2)

In [ ]:
# matriz de correlação usando Seaborn com heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlacoes, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Matriz de Correlação entre Variáveis Numéricas')
plt.show()

Para facilitar a escolha das variáveis, podemos observar especificamente a relação entre cada variável e o **consumo**.

In [ ]:
dados["consumo"].sort_values(ascending=False).round(2)

### O que observar?

Uma variável com correlação mais distante de zero pode ser uma candidata interessante para investigar por meio de regressão linear.

Neste exemplo, vamos utilizar:

- **X = temperatura máxima (`temp_max`)**
- **y = consumo de cerveja (`consumo`)**

A escolha também deve fazer sentido no contexto do problema, e não apenas pelo valor da correlação.

# Exemplo resolvido — Regressão Linear

## 7. Visualizando a relação entre as variáveis

Antes de treinar o modelo, vamos construir um gráfico de dispersão.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(dados["temp_max"], dados["consumo"], alpha=0.7)

plt.xlabel("Temperatura máxima (°C)")
plt.ylabel("Consumo de cerveja (litros)")
plt.title("Temperatura máxima x consumo de cerveja")
plt.grid(alpha=0.2)

plt.show()

Cada ponto representa um registro da base.

O gráfico ajuda a verificar visualmente se existe uma tendência entre as duas variáveis.

## 8. Definindo X e y

Em regressão linear:

- **X** representa a variável utilizada como entrada do modelo;
- **y** representa a variável que queremos estimar.

Aqui, utilizaremos a temperatura máxima para estimar o consumo.

In [ ]:
X = dados[["temp_max"]]
y = dados["consumo"]

print("X:")
display(X.head())

print("y:")
display(y.head())

## 9. Separando dados de treino e teste

Vamos utilizar:

- **80% dos registros para treino**;
- **20% dos registros para teste**.

Os dados de treino são utilizados para ajustar o modelo.  
Os dados de teste ficam separados para observar o comportamento do modelo com registros que não participaram do treinamento.

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Registros de treino:", len(X_treino))
print("Registros de teste:", len(X_teste))

## 10. Criando e treinando o modelo

`LinearRegression()` cria o modelo.

O método `fit()` utiliza os dados de treino para encontrar a reta que melhor representa a relação entre X e y.

In [ ]:
modelo = LinearRegression()
modelo.fit(X_treino, y_treino)

## 11. Coeficientes da regressão

Uma regressão linear simples pode ser representada por:

**y = aX + b**

onde:

- **b** é o intercepto;
- **a** é o coeficiente angular.

O coeficiente angular indica quanto o valor estimado de **y** tende a mudar quando **X aumenta uma unidade**.

O intercepto representa o valor estimado de **y quando X = 0**. Dependendo do problema, esse valor pode não ter uma interpretação prática.

In [ ]:
intercepto = modelo.intercept_
coeficiente = modelo.coef_[0]

print(f"Intercepto: {intercepto:.4f}")
print(f"Coeficiente angular: {coeficiente:.4f}")

### Exemplo de interpretação

Se o coeficiente angular for positivo, o modelo indica que o consumo estimado tende a aumentar quando a temperatura máxima aumenta.

Se o coeficiente angular for negativo, o modelo indica uma tendência de redução.

A interpretação deve utilizar o **valor obtido na execução do modelo** e as unidades das duas variáveis.

## 12. Fazendo previsões

In [ ]:
y_previsao = modelo.predict(X_teste)

In [ ]:
resultado = pd.DataFrame({
    "Temperatura máxima": X_teste["temp_max"].values,
    "Consumo real": y_teste.values,
    "Consumo previsto": y_previsao
})

In [ ]:
resultado.head(10)

## 13. Gráfico com os dados e a reta ajustada

Este é o tipo de gráfico solicitado na Challenge Sprint 3.

O gráfico deve conter:

- os dados;
- a reta de regressão;
- título;
- identificação dos eixos;
- legenda.

In [ ]:
x_reta = np.linspace(
    dados["temp_max"].min(),
    dados["temp_max"].max(),
    100
).reshape(-1, 1)

y_reta = modelo.predict(x_reta)

plt.figure(figsize=(9, 5))

plt.scatter(
    dados["temp_max"],
    dados["consumo"],
    alpha=0.6,
    label="Dados"
)

plt.plot(
    x_reta,
    y_reta,
    linewidth=2,
    label="Reta de regressão"
)

plt.xlabel("Temperatura máxima (°C)")
plt.ylabel("Consumo de cerveja (litros)")
plt.title("Regressão Linear: temperatura máxima x consumo")
plt.legend()
plt.grid(alpha=0.2)

plt.show()

## 14. Fazendo uma nova estimativa

Depois de treinado, o modelo também pode receber um novo valor de X.

In [ ]:
nova_temperatura = pd.DataFrame({"temp_max": [30]})

consumo_estimado = modelo.predict(nova_temperatura)[0]

print(f"Para 30 °C, o consumo estimado é {consumo_estimado:.2f} litros.")

## Métricas para avaliar o modelo

Depois de treinar um modelo de Regressão Linear, podemos utilizar algumas métricas para verificar a qualidade das previsões.

### MAE — Erro Absoluto Médio

O **MAE (Mean Absolute Error)** indica, em média, o quanto as previsões do modelo estão distantes dos valores reais.

Quanto **menor o MAE**, menores são os erros de previsão.

---

### MSE — Erro Quadrático Médio

O **MSE (Mean Squared Error)** calcula a média dos erros elevados ao quadrado.

Por elevar os erros ao quadrado, essa métrica dá **maior peso aos erros grandes**. Por isso, pode ser útil para identificar modelos que apresentam previsões muito distantes dos valores reais.

Quanto **menor o MSE**, melhor o resultado do modelo.

---

### R² — Coeficiente de Determinação

O **R²** indica quanto da variação da variável que queremos prever é explicada pelo modelo, em comparação com uma previsão baseada apenas na média.

Um valor de **R² próximo de 1** indica que o modelo explica bem a variação observada nos dados.

Exemplo:

- **R² = 0,85** → o modelo explica aproximadamente **85% da variação observada** na variável analisada.
- **R² = 0** → o modelo não melhora a explicação em relação ao uso da média.
- **R² < 0** → o modelo apresenta desempenho pior do que utilizar a média como previsão.

O R² deve ser analisado junto com métricas de erro, como **MAE e MSE**, para avaliar melhor o desempenho do modelo.

In [ ]:
mae = mean_absolute_error(y_teste, y_previsao)
mse = mean_squared_error(y_teste, y_previsao)
r2 = r2_score(y_teste, y_previsao)

print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R²: {r2:.4f}")